# BestShot — Chameleon Node Setup

Run this notebook from **Experiment > Jupyter Interface** on chameleoncloud.org.

Before running:
1. You must have an **active** `gpu_mi100` lease on CHI@TACC
2. Your SSH key must be uploaded to CHI@TACC

Run all cells top to bottom. At the end you will have a floating IP to SSH into.

## 1. Connect to your project

In [ ]:
from chi import server, context, lease
import os

context.version = "1.0"
context.choose_project()               # pick your CHI-XXXXXX project from the dropdown
context.choose_site(default="CHI@TACC")

## 2. Get your active lease

Change `YOUR_LEASE_NAME` to whatever you named your lease on Chameleon (e.g. `bestshot-serving`).

In [ ]:
LEASE_NAME = "serve_proj19_testing_lease"   # change this

l = lease.get_lease(LEASE_NAME)
l.show()   # status should say ACTIVE

## 3. Launch the bare metal server

Uses the ROCm-compatible Ubuntu image (required for AMD MI100 GPU).

**Note:** if you already have a server with this name in ERROR state, delete it first in the Horizon GUI before running this cell.

In [ ]:
username = os.getenv("USER")

s = server.Server(
    f"node-bestshot-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-ROCm"
)
s.submit(idempotent=True)
print("Server submitted — waiting for it to become active...")

## 4. Assign a floating IP so you can SSH in

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
s.refresh()
s.show(type="widget")   # note the floating IP in the Addresses row

## 5. a. (Run if repo is NOT in Jupyter Interface) Clone your repo onto the node

In [ ]:
TOKEN = "ghp_xxxx" # private repo required a token, public repo can be cloned without it

s.execute(f"git clone https://{TOKEN}@github.com/anokhimehta/bestshot.git")

## 5. b. (Run this if repo is already in Jupyter Interface) Go into project folder and pull any changes

In [ ]:
s.execute("cd bestshot && git pull")

## 6. Install Docker

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

## 7. Verify the GPU is visible

In [ ]:
s.execute("rocm-smi")

## 8. Build your serving Docker image

In [ ]:
REPO_DIR = "bestshot/serving"    # using the folder name git clone created (bestshot), and inside where the Dockerfile is (serving)

s.execute(f"docker build -t bestshot-serve -f {REPO_DIR}/docker/Dockerfile.serve {REPO_DIR}/")

## 9. Start the serving container

In [ ]:
s.execute(
    "docker run -d --rm "
    "--network host "
    "--device=/dev/kfd "
    "--device=/dev/dri "
    "--group-add video "
    "--ipc=host "
    "--name bestshot-api "
    "bestshot-serve"
)

## 10. Options for running the code:
1. running on terminal 
2. running on vs code
3. running benchmark tests on notebook (benchmark_tests_ipynb)

### Option 1: Terminal - SSH in to run benchmarks manually

From your **local terminal**, run:

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>
```

Then on the node:

1. Pull latest code
```bash
    cd bestshot
    git pull
```

2. Rebuild the image with your changes
```bash
    docker build -t bestshot-serve -f serving/docker/Dockerfile.serve serving/
```

3. Clean up old container
```bash
    docker rm -f bestshot-api
```

4. start the API server in background
```bash
    docker run -d \
      --network host \
      --name bestshot-api \
      bestshot-serve \
      uvicorn app:app --host 0.0.0.0 --port 8000
```

6. run the benchmark inside a container
```bash
    docker run --rm \
      --network host \
      bestshot-serve \
      python benchmark.py
```

or state the CONFIG_NAME
```bash
    docker run --rm \
  --network host \
  -e CONFIG_NAME=cpu_sequential \
  bestshot-serve \
  python benchmark.py
```

### Option 2: VS Code

Run the command below first and then follow the rest of the steps

In [ ]:
# Add ability to run commands through run.sh
# add the "executable" permission, which tells the OS "this file is allowed to be run as a script."
s.execute("chmod +x bestshot/serving/run.sh")

# terminal equivalent: chmod +x serving/run.sh

1. Connect to ssh from VS Code:

    * Press Cmd + Shift + P
    * Type: Remote-SSH: Connect to Host
    * Select: chameleon-node

    Then, VS Code will:
    * SSH into the machine
    * Open a remote workspace
    * Let you edit files as if they’re local


2. To set up env,  open terminal in VS Code:

    Once connected open a new terminal and run:
    ```bash 
        code . 
    ```
     Now you’re running commands on the remote machine, not your laptop. And you are able to see your repo on the application.

3. Set up the node: 

    a. Confirm you are in 
    ```bash
        cc@node-bestshot-lg3323-nyu-edu:~/bestshot$
    ```

    b. Pull latest code
    ```bash
        git pull
    ```

    c. Rebuild Docker image with changes (e.g. change: Python files, dependencier, Dockerfile):
    ```bash
        docker build -t bestshot-serve -f serving/docker/Dockerfile.serve serving/
    ```

    d. Remove old container
    ```bash
        docker rm -f bestshot-api
    ```

    e. Start API and run benchmark, examples:
    ```bash
        ./serving/run.sh cpu_sequential
        ./serving/run.sh cpu_batch
        ./serving/run.sh gpu_batch
        ./serving/run.sh gpu_sequential
        ./serving/run.sh gpu_onnx
    ```





## Optional - helper code blocks during implemention/testing 

Notebook code not terminal

In [ ]:
# Push and rebuild
s.execute("cd bestshot && git pull")
s.execute("docker build -t bestshot-serve -f bestshot/serving/docker/Dockerfile.serve bestshot/serving/")
s.execute("docker rm -f bestshot-api")

## 11. Delete Resources

In [ ]:
# Stop the API container if it's still running
s.execute("docker stop bestshot-api")

In [ ]:
# Delete the container 
s.execute("docker rm bestshot-api")

In [ ]:
# Shut down the bare metal server to release it back
s.delete()